In [9]:
from pathlib import Path
import re
import pandas as pd

pattern = re.compile(
    r"^\s*\d+\s+(\d{4}-\d{2}-\d{2}\s+\d{2}:\d{2}:\d{2})\s+([-+]?\d*\.?\d+)\s+([-+]?\d*\.?\d+)\s*$"
    )

def load_solar_txt(file_path: Path) -> pd.DataFrame:
    rows = []
    for line in file_path.read_text(encoding="utf-8").splitlines():
        match = pattern.match(line)
        if match:
            rows.append(match.groups())

    df = pd.DataFrame(rows, columns=["time", "predicted_kw", "temperature_2m"])
    df["time"] = pd.to_datetime(df["time"])
    for col in ["predicted_kw", "temperature_2m"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df["source"] = file_path.stem.split()[0]
    return df

txt_files = sorted(Path("data/validate").glob("*.txt"))
df_all = pd.concat([load_solar_txt(fp) for fp in txt_files], ignore_index=True)

df_all.head()

,time,predicted_kw,temperature_2m,source
0,2026-03-09 00:00:00,0.0,9.6,sun1_kw
1,2026-03-09 00:15:00,0.0,9.3,sun1_kw
2,2026-03-09 00:30:00,0.0,9.0,sun1_kw
3,2026-03-09 00:45:00,0.0,8.7,sun1_kw
4,2026-03-09 01:00:00,0.0,8.5,sun1_kw


In [10]:
# 15-minute values: kWh per row = kW * 0.25 hours
df_all["energy_kwh"] = df_all["predicted_kw"] * 0.25

energy_by_source = df_all.groupby("source", as_index=False)["energy_kwh"].sum()
total_kwh = energy_by_source["energy_kwh"].sum()

print(energy_by_source)
print(f"Total energy (all txt files): {total_kwh:.3f} kWh")

    source  energy_kwh
0  sun1_kw   16.281972
1  sun2_kw    5.913287
Total energy (all txt files): 22.195 kWh


In [11]:
df_all

,time,predicted_kw,temperature_2m,source,energy_kwh
0,2026-03-09 00:00:00,0.000000,9.6,sun1_kw,0.000000
1,2026-03-09 00:15:00,0.000000,9.3,sun1_kw,0.000000
2,2026-03-09 00:30:00,0.000000,9.0,sun1_kw,0.000000
3,2026-03-09 00:45:00,0.000000,8.7,sun1_kw,0.000000
4,2026-03-09 01:00:00,0.000000,8.5,sun1_kw,0.000000
...,...,...,...,...,...
139,2026-03-09 10:45:00,1.951513,12.7,sun2_kw,0.487878
140,2026-03-09 11:00:00,2.048293,13.4,sun2_kw,0.512073
141,2026-03-09 11:15:00,2.132496,13.9,sun2_kw,0.533124
142,2026-03-09 11:30:00,2.208099,14.4,sun2_kw,0.552025
